# Lecture 5 — Class Exercise
## Distribution Charts: Airbnb London

> **Push to:** `week05/lecture05_exercise.ipynb`

**Rules:**
1. Cap price outliers at 95th percentile — annotate this
2. Every chart has a **median/mean reference line** with annotation
3. Insight title names the distribution shape or key finding
4. Colour has meaning — don't use colour just for decoration

---


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Airbnb London Listings

df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


In [ ]:
p95 = df['price'].quantile(0.95)
df_cap = df[df['price'] <= p95]
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


## Task 1 — Histogram: price by room type (overlapping distributions)

**What to build:** A histogram showing price distributions for **Entire home/apt vs Private room** (exclude Shared room — too few observations) overlaid on the same chart.

**Requirements:**
- Both room types on the same chart (use `color='room_type'`)
- `barmode='overlay'` with `opacity=0.6` so both distributions are visible
- A vertical line for the median of EACH room type, differently coloured
- Insight title comparing the two distributions

In [ ]:
df_t1 = df_cap.loc[df_cap['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

# medians for the reference lines and the title
med_home = df_t1.loc[df_t1['room_type'] == 'Entire home/apt', 'price'].median()
med_room = df_t1.loc[df_t1['room_type'] == 'Private room', 'price'].median()

fig = px.histogram(
    data_frame=df_t1,
    x='price',
    color='room_type',
    barmode='overlay',                     
    opacity=0.6,                           
    nbins=50,
    color_discrete_map={'Entire home/apt': '#2E75B6', 'Private room': '#70AD47'},
    labels={'price': 'Nightly Price (£)', 'room_type': ''},
    height=500,
    template='plotly_white'
)

fig.add_vline(x=med_home, line_dash='dash', line_color='#2E75B6', line_width=3,
              annotation=dict(text=f'Entire home median: £{med_home:.0f}',
                              font=dict(color='#2E75B6', size=13),
                              xanchor='left', yanchor='top', xshift=8))

fig.add_vline(x=med_room, line_dash='dash', line_color='#70AD47', line_width=3,
              annotation=dict(text=f'Private room median: £{med_room:.0f}',
                              font=dict(color='#70AD47', size=13),
                              xanchor='left', yanchor='bottom', xshift=8, y=0.85, yref='paper'))

fig.update_layout(
    title=f'Both price distributions are right-skewed — entire homes cost {med_home / med_room:.1f}x more than private rooms (median)',
    font=dict(family='Arial', size=13),
    yaxis=dict(title='Number of Listings'),
    xaxis=dict(showgrid=False, title=f'Nightly Price (£) - Data Capped at 95th percentile (£{p95:.0f})'),
    legend=dict(orientation='h', x=0, y=1.12),
    margin=dict(t=100)
)
fig.show()


**Task 1 — what the code does**
- `isin([...])` keeps only Entire home/apt and Private room, using `df_cap`, the data with prices capped at the 95th percentile.
- `.median()` gives the median price of each room type, for the two lines and the title.
- `color='room_type'` draws one histogram per room type, with a meaningful colour each.
- `barmode='overlay'` puts both histograms on the same axis, and `opacity=0.6` makes them transparent so you can see where they overlap.
- `add_vline(...)` draws a dashed median line for each room type, in the same colour as its bars, with a text label.
- The x-axis title says the data is capped at the 95th percentile, so the cap is disclosed.
- The title names the shape (right-skewed: a long tail of expensive listings) and compares the two medians.


## Task 2 — Box plot: listing activity by borough

**What to build:** A **horizontal box plot** comparing listing activity (reviews per month) across London boroughs — reviews per month is a proxy for how frequently a listing is booked.

**Requirements:**
- Horizontal orientation (borough names are long)
- Sorted by median reviews per month (most active at top)
- Highlight the **two most active** boroughs in a different colour
- Outliers shown as individual points
- Insight title naming the two busiest boroughs

> 💡 Some listings have zero reviews — these are new or inactive listings. Filter them out with before plotting

In [ ]:
# Task 2 — Box plot: reviews per month by borough
# remove listings with 0 reviews (new or inactive)
df_t2 = df.loc[df['reviews_per_month'] > 0].copy()

med_order = df_t2.groupby('neighbourhood')['reviews_per_month'].median().sort_values(ascending=False).index.tolist()
top2 = med_order[:2]                       

colour_map = {b: ('crimson' if b in top2 else 'slategrey') for b in med_order}

fig = px.box(
    data_frame=df_t2,
    x='reviews_per_month',
    y='neighbourhood',                     #
    color='neighbourhood',
    color_discrete_map=colour_map,
    category_orders={'neighbourhood': med_order},   
    points='outliers',                     
    labels={'reviews_per_month': 'Reviews per Month', 'neighbourhood': ''},
    height=800,
    template='plotly_white'
)

overall_med = df_t2['reviews_per_month'].median()
fig.add_vline(x=overall_med, line_dash='dash', line_color='grey', line_width=2,
              annotation=dict(text=f'London median: {overall_med:.1f}',
                              font=dict(color='grey', size=13),
                              xanchor='left', yanchor='bottom', xshift=8))

fig.update_layout(
    title=f'{top2[0]} and {top2[1]} are the busiest boroughs — highest median reviews per month',
    font=dict(family='Arial', size=12),
    xaxis=dict(title='Reviews per Month (listings with 0 reviews removed)'),
    yaxis=dict(showgrid=False),
    showlegend=False,
    boxmode='overlay'                     
)
fig.show()


**Task 2 — what the code does**
- `df['reviews_per_month'] > 0` removes new or inactive listings with no reviews.
- `groupby(...).median().sort_values(ascending=False)` ranks the boroughs by median activity, and `med_order[:2]` takes the top two.
- `colour_map` makes the top two crimson and every other borough grey (grey + highlight, as in the notes).
- `y='neighbourhood'` makes the boxes horizontal, so the long borough names are readable.
- `category_orders` keeps the boroughs sorted, with the most active at the top.
- `points='outliers'` shows the outliers as individual dots.
- `add_vline(...)` draws a reference line at the London-wide median, with a label.
- `boxmode='overlay'` puts each box in the middle of its row; without it, the coloured boxes get shifted sideways.
- `showlegend=False` hides the legend, because the title already names the highlighted boroughs.
